In [ ]:
!pip install langchain_openai
!pip install langchain_core

In [ ]:
#import the necessary libraries:
from typing import TypedDict, List, Optional, Union
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool, tool
from langchain_core.messages import HumanMessage , SystemMessage , AIMessage , ToolMessage, ToolCall , BaseMessage

In [ ]:
## Define a State Schema
#Create a TypedDict to represent the agent’s state, including fields for the user query, instructions, message history, and any pending tool calls.
class AgentState(TypedDict):
    user_query: str  # The current user query being processed
    instructions: str  # System instructions for the agent
    messages: List[dict]  # List of conversation messages
    current_tool_calls: Optional[List[ToolCall]]

In [ ]:
@tool
def get_games(num_games:int=1, top:bool=True) -> str:
    """
    Returns the top or bottom N games with highest or lowest scores.
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88}
    ]
    # Sort the games list by Score
    # If top is True, descending order
    sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)

    # Return the N games
    return sorted_games[:num_games]


In [ ]:
tools = [get_games]

In [ ]:
## Create the Steps
#**Prepare Messages**: Build the message list for the LLM.
def prepare_messages_step(state: AgentState) -> AgentState:
    """Step logic: Prepare messages for LLM consumption"""

    messages = [
        SystemMessage(content=state["instructions"]),
        HumanMessage(content=state["user_query"])
    ]

    return {
        "messages": messages
    }


In [ ]:
#**LLM Step**: Call the language model and check for tool calls.
def llm_step(state: AgentState) -> AgentState:
    """Step logic: Process the current state through the LLM"""

    # Initialize LLM
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.3,
        tools=tools,
    )

    response = llm.invoke(state["messages"])
    # Ensure tool_calls is an empty list if no tool calls are present
    tool_calls = response.tool_calls if response.tool_calls else []

    # Create AI message with content and tool calls
    ai_message = AIMessage(content=response.content, tool_calls=tool_calls)

    return {
        "messages": state["messages"] + [ai_message],
        "current_tool_calls": tool_calls
    }

In [ ]:
#**Tool Step**: Execute any tool calls and update the state.
def tool_step(state: AgentState) -> AgentState:
    """Step logic: Execute any pending tool calls"""
    tool_calls = state["current_tool_calls"] or []
    tool_messages = []

    for call in tool_calls:
        # Access tool call data correctly
        function_name = call.function.name
        function_args = json.loads(call.function.arguments)
        tool_call_id = call.id
        # Find the matching tool
        tool = next((t for t in tools if t.name == function_name), None)
        if tool:
            result = tool(**function_args)
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(result),
                    tool_call_id=tool_call_id,
                    name=function_name,
                )
            )

# Clear tool calls and add results to messages
    return {
        "messages": state["messages"] + tool_messages,
        "current_tool_calls": None
    }

In [ ]:
## Build and Connect the State Machine

#Add your steps to the state machine, and connect them with transitions. Use conditional routing to decide whether to call tools or terminate, and loop as needed.

workflow = StateMachine[AgentState](AgentState)

# Create steps
entry = EntryPoint[AgentState]()
message_prep = Step[AgentState]("message_prep", prepare_messages_step)
llm_processor = Step[AgentState]("llm_processor", llm_step)
tool_executor = Step[AgentState]("tool_executor", tool_step)
termination = Termination[AgentState]()

workflow.add_steps(
    [
        entry,
        message_prep,
        llm_processor,
        tool_executor,
        termination
    ]
)


# Add transitions
workflow.connect(entry, message_prep)
workflow.connect(message_prep, llm_processor)

In [ ]:
import os
from langchain_core.utils.function_calling import convert_to_openai_tool

# Transition based on whether there are tool calls
def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
    """Transition logic: Check if there are tool calls"""
    if state.get("current_tool_calls"):
        return tool_executor
    return termination

# Routing: If tool calls -> tool_executor
workflow.connect(
    source=llm_processor,
    targets=[tool_executor, termination],
    condition=check_tool_calls
)

# Looping: Go back to llm after tool execution
workflow.connect(
    source=tool_executor,
    targets=llm_processor
)

# Fix for AttributeError: 'dict' object has no attribute 'function'
# The tool_step function from cell oYAd2DHOTtkp is being called,
# but 'ToolCall' objects are being passed as dictionaries.
# Redefine tool_step locally to handle dictionary-like ToolCalls.
def fixed_tool_step(state: AgentState) -> AgentState:
    """Step logic: Execute any pending tool calls, handling dict-like ToolCalls"""
    tool_calls = state["current_tool_calls"] or []
    tool_messages = []

    for call in tool_calls:
        # Access tool call data using dictionary keys
        # ToolCall objects are converted to dicts in transit, with 'name' and 'args' at top level
        function_name = call['name'] # Corrected: Access 'name' directly
        function_args = call['args'] # Corrected: 'args' is already a dict, no json.loads needed
        tool_call_id = call['id']

        # Find the matching tool
        # 'tools' list should be available from global scope (cell 2OeGZLfdThcD)
        tool = next((t for t in tools if t.name == function_name), None)
        if tool:
            # Corrected: Call the tool's invoke method
            result = tool.invoke(function_args)
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(result),
                    tool_call_id=tool_call_id,
                    name=function_name,
                )
            )
    return {
        "messages": state["messages"] + tool_messages,
        "current_tool_calls": None
    }

# Update the 'tool_executor' step in the workflow to use the fixed_tool_step
# Assuming 'tool_executor' is a Step object already added to the workflow
# and can be accessed via workflow.steps dictionary. The tool_executor object
# itself was created in cell d31hVfpZTz8c and is available globally.
tool_executor.func = fixed_tool_step

# Fix for ValidationError: AIMessage tool_calls Input should be a valid list
# Redefine llm_step locally to ensure tool_calls is always a list.
def fixed_llm_step(state: AgentState) -> AgentState:
    """Step logic: Process the current state through the LLM, ensuring tool_calls is a list"""

    # Convert LangChain tools to OpenAI format
    openai_tools = [convert_to_openai_tool(t) for t in tools]

    # Initialize LLM (assuming 'tools' and 'ChatOpenAI' are accessible from global scope)
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.3,
        tools=openai_tools,
    )

    response = llm.invoke(state["messages"])
    # Ensure tool_calls is an empty list if no tool calls are present, instead of None
    tool_calls = response.tool_calls if response.tool_calls else []

    # Create AI message with content and tool calls
    ai_message = AIMessage(content=response.content, tool_calls=tool_calls)

    return {
        "messages": state["messages"] + [ai_message],
        "current_tool_calls": tool_calls
    }

# Update the 'llm_processor' step in the workflow to use the fixed_llm_step
llm_processor.func = fixed_llm_step

## Run the Workflow
initial_state: AgentState = {
    "user_query": "What's the best game in the dataset?",
    "instructions": "You can bring insights about a game dataset based on users questions",
    "messages": [],
}

# Set your OpenAI API key here. Replace 'YOUR_OPENAI_API_KEY' with your actual key.
# For production, consider using Colab secrets or a .env file.
os.environ["OPENAI_API_KEY"] = ""

run_object = workflow.run(initial_state)
run_object.get_final_state()["messages"]

In [ ]:
from typing import TypedDict, List, Optional, Union
import json
from dotenv import load_dotenv

# Langchain related imports
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage, ToolCall
from langchain_core.tools import Tool, tool
from langchain_core.utils.function_calling import convert_to_openai_tool # Added import

# The following classes (StateMachine, Step, EntryPoint, Termination, Run)
# were originally intended to be imported from a custom 'lib' directory (e.g., from lib.state_machine).
# Since 'lib' module was not found, these classes are currently undefined.
# For the code to run, basic placeholder classes are provided below. In a real application,
# you would replace these with actual implementations or ensure their correct import.
class StateMachine[State]:
    def __init__(self, state_type):
        self.state_type = state_type
        self.steps = {}
        self.transitions = {}
        self.entry_point = None

    def add_steps(self, steps):
        for step in steps:
            self.steps[step.name] = step
            if isinstance(step, EntryPoint):
                self.entry_point = step

    def connect(self, source, targets, condition=None):
        if not isinstance(targets, list):
            targets = [targets]
        self.transitions[source.name] = (targets, condition)

    def run(self, initial_state):
        current_state = initial_state
        current_step_name = self.entry_point.name if self.entry_point else None

        # Simple execution loop for demonstration
        while current_step_name and current_step_name in self.steps:
            current_step = self.steps[current_step_name]
            if isinstance(current_step, EntryPoint):
                # EntryPoint does not modify state but leads to the next step
                pass
            elif isinstance(current_step, Termination):
                break
            else:
                updated_state_delta = current_step.execute(current_state)
                current_state.update(updated_state_delta)

            # Determine next step
            if current_step_name in self.transitions:
                targets, condition = self.transitions[current_step_name]
                if condition:
                    next_step_or_name = condition(current_state)
                    if isinstance(next_step_or_name, Step):
                        current_step_name = next_step_or_name.name
                    else: # Assuming it's a step name string
                        current_step_name = next_step_or_name
                else:
                    # Default to the first target if no condition
                    current_step_name = targets[0].name
            else:
                # No transition defined, attempt to go to a default next step (e.g., Termination)
                if 'termination' in self.steps:
                    current_step_name = 'termination'
                else:
                    break # No more steps or termination

        return Run(current_state) # Return a Run object containing the final state

class Step[State]:
    def __init__(self, name: str, func):
        self.name = name
        self.func = func

    def execute(self, state: State) -> State:
        return self.func(state)

class EntryPoint[State](Step[State]):
    def __init__(self):
        super().__init__("entry_point", lambda s: s) # Entry point usually doesn't modify state

class Termination[State](Step[State]):
    def __init__(self):
        super().__init__("termination", lambda s: s) # Termination usually doesn't modify state

class Run:
    def __init__(self, final_state):
        self._final_state = final_state

    def get_final_state(self):
        return self._final_state

# The 'LLM' class in Agent._llm_step is likely a custom abstraction.
# Since ChatOpenAI is being used elsewhere, LLM is aliased to ChatOpenAI for consistency.
# If a custom LLM wrapper is truly needed, its definition should be provided.
class LLM(ChatOpenAI):
    pass

load_dotenv()
## Define a State Schema
#Create a TypedDict to represent the agent’s state, including fields for the user query, instructions, message history, and any pending tool calls.
# AgentState definition has been moved to cell vGHl7XxaTB4p

## Define the Tools you will use


@tool
def get_games(num_games:int=1, top:bool=True) -> str:
    """
    Returns the top or bottom N games with highest or lowest scores.
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88}
    ]
    # Sort the games list by Score
    # If top is True, descending order
    sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)

    # Return the N games
    return sorted_games[:num_games]


tools = [get_games]
## Create the Steps


#**Prepare Messages**: Build the message list for the LLM.
def prepare_messages_step(state: AgentState) -> AgentState:
    """Step logic: Prepare messages for LLM consumption"""

    messages = [
        SystemMessage(content=state["instructions"]),
        HumanMessage(content=state["user_query"])
    ]

    return {
        "messages": messages
    }


#**LLM Step**: Call the language model and check for tool calls.
def llm_step(state: AgentState) -> AgentState:
    """Step logic: Process the current state through the LLM"""

    # Convert LangChain tools to OpenAI format
    openai_tools = [convert_to_openai_tool(t) for t in tools]

    # Initialize LLM
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.3,
        tools=openai_tools,
    )

    response = llm.invoke(state["messages"])
    tool_calls = response.tool_calls if response.tool_calls else None

    # Create AI message with content and tool calls
    ai_message = AIMessage(content=response.content, tool_calls=tool_calls)

    return {
        "messages": state["messages"] + [ai_message],
        "current_tool_calls": tool_calls
    }


#**Tool Step**: Execute any pending tool calls and update the state.
def tool_step(state: AgentState) -> AgentState:
    """Step logic: Execute any pending tool calls"""
    tool_calls = state["current_tool_calls"] or []
    tool_messages = []

    for call in tool_calls:
        # Access tool call data correctly
        function_name = call.function.name
        function_args = json.loads(call.function.arguments)
        tool_call_id = call.id
        # Find the matching tool
        tool = next((t for t in tools if t.name == function_name), None)
        if tool:
            result = tool(**function_args)
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(result),
                    tool_call_id=tool_call_id,
                    name=function_name,
                )
            )

# Clear tool calls and add results to messages
    return {
        "messages": state["messages"] + tool_messages,
        "current_tool_calls": None
    }

## Build and Connect the State Machine

#Add your steps to the state machine, and connect them with transitions. Use conditional routing to decide whether to call tools or terminate, and loop as needed.

workflow = StateMachine[AgentState](AgentState)

# Create steps
entry = EntryPoint[AgentState]()
message_prep = Step[AgentState]("message_prep", prepare_messages_step)
llm_processor = Step[AgentState]("llm_processor", llm_step)
tool_executor = Step[AgentState]("tool_executor", tool_step)
termination = Termination[AgentState]()

workflow.add_steps(
    [
        entry,
        message_prep,
        llm_processor,
        tool_executor,
        termination
    ]
)
# Add transitions
workflow.connect(entry, message_prep)
workflow.connect(message_prep, llm_processor)

# Transition based on whether there are tool calls
def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
    """Transition logic: Check if there are tool calls"""
    if state.get("current_tool_calls"):
        return tool_executor
    return termination

# Routing: If tool calls -> tool_executor
workflow.connect(
    source=llm_processor,
    targets=[tool_executor, termination],
    condition=check_tool_calls
)

# Looping: Go back to llm after tool execution
workflow.connect(
    source=tool_executor,
    targets=llm_processor
)
## Run the Workflow
initial_state: AgentState = {
    "user_query": "What's the best game in the dataset?",
    "instructions": "You can bring insights about a game dataset based on users questions",
    "messages": [],
}
run_object = workflow.run(initial_state)
run_object.get_final_state()["messages"]


## Optional
#Create an Agent class to encapsulate State Machine logic. Then try adding more tools, and experiment with different user queries to see how the workflow adapts.
class Agent:
    def __init__(self,
                 model_name: str,
                 instructions: str,
                 tools: List[Tool] = None,
                 temperature: float = 0.7):
        """
        Initialize an Agent instance

        Args:
            model_name: Name/identifier of the LLM model to use
            instructions: System instructions for the agent
            tools: Optional list of tools available to the agent
            temperature: Temperature parameter for LLM (default: 0.7)
        """
        self.instructions = instructions
        self.tools = tools if tools else []
        self.model_name = model_name
        self.temperature = temperature

        # Initialize state machine
        self.workflow = self._create_state_machine()

    def _prepare_messages_step(self, state: AgentState) -> AgentState:
        """Step logic: Prepare messages for LLM consumption"""

        messages = [
            SystemMessage(content=state["instructions"]),
            HumanMessage(content=state["user_query"])
        ]

        return {
            "messages": messages
        }

    def _llm_step(self, state: AgentState) -> AgentState:
        """Step logic: Process the current state through the LLM"""

        # Convert LangChain tools to OpenAI format
        openai_tools = [convert_to_openai_tool(t) for t in self.tools]

        # Initialize LLM
        llm = ChatOpenAI(
            model=self.model_name,
            # api_key = "
            temperature=self.temperature,
            tools=openai_tools
        )

        response = llm.invoke(state["messages"])
        tool_calls = response.tool_calls if response.tool_calls else None

        # Create AI message with content and tool calls
        ai_message = AIMessage(content=response.content, tool_calls=tool_calls)

        return {
            "messages": state["messages"] + [ai_message],
            "current_tool_calls": tool_calls
        }

    def _tool_step(self, state: AgentState) -> AgentState:
        """Step logic: Execute any pending tool calls"""
        tool_calls = state["current_tool_calls"] or []
        tool_messages = []

        for call in tool_calls:
            # Access tool call data correctly
            function_name = call.function.name
            function_args = json.loads(call.function.arguments)
            tool_call_id = call.id
            # Find the matching tool
            tool = next((t for t in self.tools if t.name == function_name), None)
            if tool:
                result = tool(**function_args)
                tool_messages.append(
                    ToolMessage(
                        content=json.dumps(result),
                        tool_call_id=tool_call_id,
                        name=function_name,
                    )
                )

        # Clear tool calls and add results to messages
        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None
        }

    def _create_state_machine(self) -> StateMachine[AgentState]:
        """Create the internal state machine for the agent"""
        machine = StateMachine[AgentState](AgentState)

        # Create steps
        entry = EntryPoint[AgentState]()
        message_prep = Step[AgentState]("message_prep", self._prepare_messages_step)
        llm_processor = Step[AgentState]("llm_processor", self._llm_step)
        tool_executor = Step[AgentState]("tool_executor", self._tool_step)
        termination = Termination[AgentState]()

        machine.add_steps([entry, message_prep, llm_processor, tool_executor, termination])

        # Add transitions
        machine.connect(entry, message_prep)
        machine.connect(message_prep, llm_processor)

        # Transition based on whether there are tool calls
        def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
            """Transition logic: Check if there are tool calls"""
            if state.get("current_tool_calls"):
                return tool_executor
            return termination

        machine.connect(llm_processor, [tool_executor, termination], check_tool_calls)
        machine.connect(tool_executor, llm_processor)  # Go back to llm after tool execution

        return machine

    def invoke(self, query: str) -> Run:
        """
        Run the agent on a query

        Args:
            query: The user's query to process

        Returns:
            The final run object after processing
        """

        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages": [],
        }

        run_object = self.workflow.run(initial_state)

        return run_object

@tool
def power(base:float, exponent:float):
    """Exponentatiation: base to the power of exponent"""

    return base ** exponent
@tool
def multiply(number_a:float, number_b:float):
    """Multiplication: number_a times number_b"""
    return number_a * number_b

tools = [power, multiply]
math_agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=(
        "You're an AI Agent very good with math operations "
        "You can answer multistep questions by sequentially calling functions. "
        "You follow a pattern of of Thought and Action. "
        "Create a plan of execution: "
        "- Use Thought to describe your thoughts about the question you have been asked. "
        "- Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly."
        "When you think it's over, return the answer "
        "Never try to respond directly if the question needs a tool. "
        "But if you don't have a tool available, you can respond directly. "
        f"The actions you have are the Tools: {tools}. \n"
    )
)
run_object = math_agent.invoke(
    query="What's 3 to the power of 2? Take the result, then multiply it by 5.",
)
run_object.get_final_state()["messages"]
